# 6) Unit commitment — Exercises

**Goals:** count start-ups and measure the integrality gap, build a unit-commitment MILP with a variable coal minimum output and match PyPSA, compute uplift payments from fixed-commitment prices, and add minimum up and down times — including the plant's history before the day.

The setup cell holds the lecture's day and a `pypsa_day` helper: unit commitment by default, the fixed-commitment LP when you pass a status table, and optional minimum up/down times. Power in MW, money in €.

**Submit** your notebook for review before opening the solution in `solutions/`.

In [ ]:
import logging
import warnings

import linopy
import numpy as np
import pandas as pd
import pypsa
import xarray as xr

logging.getLogger("pypsa").setLevel(logging.ERROR)
logging.getLogger("linopy").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=FutureWarning)

T = 24
HOURS = np.arange(T)
LOAD = 1900 + 400 * np.sin(np.pi * (HOURS - 6) / 12).clip(0) + 250 * np.exp(-((HOURS - 19) / 2.0) ** 2)
FREE = 2200 * 0.8 * np.clip(np.sin(np.pi * (HOURS - 6) / 13), 0, None) + 400.0
PLANTS = pd.DataFrame({"p_nom": [900, 900, 500], "mc": [60, 95, 150], "p_min_pu": [0.4, 0.3, 0.1],
                       "start_up_cost": [20_000, 8_000, 1_000], "on_before": [1, 0, 0]},
                      index=pd.Index(["coal", "ccgt", "ocgt"], name="gen"))


def pypsa_day(plants=PLANTS, status_fixed=None, min_up=None, min_down=None, linearized=False):
    """The lecture's day in PyPSA. Unit commitment by default; with status_fixed, the fixed-commitment LP;
    linearized=True solves PyPSA's LP relaxation of the unit commitment.
    min_up / min_down: dicts {unit: hours}. The history before the day is long (24 h on or off)."""
    n = pypsa.Network()
    n.set_snapshots(HOURS)
    n.add("Bus", "HR")
    n.add("Load", "load", bus="HR", p_set=LOAD)
    n.add("Generator", "free", bus="HR", p_nom=FREE.max(), p_max_pu=FREE / FREE.max(), marginal_cost=0)
    for gen, r in plants.iterrows():
        if status_fixed is None:
            extra = {}
            if min_up and gen in min_up:
                extra["min_up_time"] = min_up[gen]
            if min_down and gen in min_down:
                extra["min_down_time"] = min_down[gen]
            n.add("Generator", gen, bus="HR", p_nom=r.p_nom, marginal_cost=r.mc, committable=True,
                  p_min_pu=r.p_min_pu, start_up_cost=r.start_up_cost,
                  up_time_before=24 if r.on_before else 0, down_time_before=0 if r.on_before else 24, **extra)
        else:
            on = status_fixed[gen].to_numpy()
            n.add("Generator", gen, bus="HR", p_nom=r.p_nom, marginal_cost=r.mc, p_max_pu=on, p_min_pu=on * r.p_min_pu)
    n.optimize(solver_name="highs", log_to_console=False, progress=False, linearized_unit_commitment=linearized)
    return n

### Warm-ups

**1. Counting start-ups**

Write `count_starts(status, on_before)` for a status table (rows: hours, columns: units, values 0/1) and the status before the day (Series). Return the number of starts per unit. A unit that was off before the day and is on in hour 0 has started.

In [ ]:
def count_starts(status: pd.DataFrame, on_before: pd.Series) -> pd.Series:
    """Number of start-ups per unit over the horizon."""
    ...  # YOUR CODE HERE

In [ ]:
n_uc = pypsa_day()
lecture_status = n_uc.generators_t.status[PLANTS.index].astype(int)
starts = count_starts(lecture_status, PLANTS.on_before)
assert starts.to_dict() == {"coal": 0, "ccgt": 2, "ocgt": 0}, "as in the lecture"
toy = pd.DataFrame({"a": [0, 1, 1, 0, 1], "b": [1, 1, 0, 0, 0]})
assert count_starts(toy, pd.Series({"a": 1, "b": 0})).to_dict() == {"a": 2, "b": 1}, "a restarts at hours 1 and 4"
starts

**2. The integrality gap**

Write `integrality_gap(milp_cost, relaxed_cost)` returning (absolute gap in €, gap relative to the MILP). Run PyPSA's unit commitment with and without `linearized_unit_commitment=True` — which, for this model, gives exactly the plain LP relaxation of the lecture.

In [ ]:
def integrality_gap(milp_cost: float, relaxed_cost: float) -> tuple[float, float]:
    """Absolute (EUR) and relative gap between a MILP and its LP relaxation."""
    ...  # YOUR CODE HERE

In [ ]:
n_relaxed = pypsa_day(linearized=True)
gap, rel = integrality_gap(n_uc.objective, n_relaxed.objective)
assert np.isclose(n_uc.objective, 1_880_184.24, atol=0.01) and np.isclose(gap, 34_464, atol=1)
assert np.isclose(rel, 0.0183, atol=1e-4)
print(f"integrality gap {gap:,.0f} EUR ({rel:.2%})")

### Core

**3. Your own unit commitment, with a knob**

Write `unit_commitment(coal_p_min_pu)` — the lecture's Linopy MILP with coal's minimum stable output (as a share of its capacity) as a parameter. Return a dict with the cost, the status table (DataFrame, 0/1) and the curtailed solar (MWh). Check it against PyPSA for 0.4 (the lecture) and 0.2 (a retrofitted, more flexible plant). What is the flexibility worth?

In [ ]:
def unit_commitment(coal_p_min_pu: float) -> dict:
    """Unit-commitment MILP of the day with a given coal minimum stable output (share of p_nom)."""
    ...  # YOUR CODE HERE

In [ ]:
FLEXIBLE = PLANTS.copy()
FLEXIBLE.loc["coal", "p_min_pu"] = 0.2
for share, plants in [(0.4, PLANTS), (0.2, FLEXIBLE)]:
    assert np.isclose(unit_commitment(share)["cost"], pypsa_day(plants).objective, rtol=1e-9), share
stiff, flexible = unit_commitment(0.4), unit_commitment(0.2)
assert flexible["cost"] < stiff["cost"] and flexible["curtailed"] < stiff["curtailed"] / 5
print(f"coal minimum 40 %: cost {stiff['cost']:,.0f} EUR, curtailed {stiff['curtailed']:,.0f} MWh")
print(f"coal minimum 20 %: cost {flexible['cost']:,.0f} EUR, curtailed {flexible['curtailed']:,.0f} MWh")
print(f"value of the retrofit: {stiff['cost'] - flexible['cost']:,.0f} EUR on this day")

**4. Uplift**

Write `uplift(n_fixed, status, plants)` for a solved fixed-commitment network, its status table and the plant data: per unit, the market margin (price − marginal cost) × output, the start-up costs, the profit, and the uplift needed to make it whole. Check the lecture's 16,000 € for the CCGT, then compute it for the flexible-coal schedule.

In [ ]:
def uplift(n_fixed, status: pd.DataFrame, plants: pd.DataFrame = PLANTS) -> pd.DataFrame:
    """Market margin, start-up costs, profit and uplift (EUR) per unit at fixed-commitment prices."""
    ...  # YOUR CODE HERE

In [ ]:
lecture = uplift(pypsa_day(status_fixed=lecture_status), lecture_status)
assert np.isclose(lecture.loc["ccgt", "uplift"], 16_000) and np.isclose(lecture.loc["coal", "profit"], 333_000)
flex_status = flexible["status"]
n_flex = pypsa_day(FLEXIBLE, status_fixed=flex_status)
flex = uplift(n_flex, flex_status, FLEXIBLE)
print("lecture schedule (coal minimum 40 %):")
print(lecture.round(0))
print("\nflexible coal (minimum 20 %):")
print(flex.round(0))
print("\nprices, flexible coal:", (n_flex.buses_t.marginal_price["HR"].round(1) + 0.0).tolist())

**5. Interpretation**

Making coal more flexible changes the curtailment, the total cost and coal's own margin. Explain each change from the prices. Why does the CCGT still need an uplift — and why is a start-up cost a *non-convex* cost that breaks the link between marginal prices and cost recovery that held in every LP so far?

*Your answer here.*

### Challenge

**6. Minimum up and down times**

Real units must stay on for a minimum time once started, and off for a minimum time once stopped. With a start variable s and a stop variable d (u[t] − u[t−1] = s[t] − d[t]) they are:

- **min up time UT**: Σ_{k = t−UT+1}^{t} s[k] ≤ u[t]
- **min down time DT**: Σ_{k = t−DT+1}^{t} d[k] ≤ 1 − u[t]

Write `uc_min_times(min_up, min_down)` with these constraints (dicts {unit: hours}; the history before the day is long, so it imposes nothing) and match PyPSA's `min_up_time` / `min_down_time` for the three cases below. Which cases change the schedule?

In [ ]:
def uc_min_times(min_up: dict, min_down: dict) -> dict:
    """Unit commitment with minimum up/down times. Returns cost (EUR) and status (DataFrame)."""
    ...  # YOUR CODE HERE

In [ ]:
cases = {"CCGT 6 h up and down": ({"ccgt": 6}, {"ccgt": 6}),
         "CCGT 12 h down": ({}, {"ccgt": 12}),
         "CCGT 12 h up": ({"ccgt": 12}, {})}
rows = {}
for name, (up, down) in cases.items():
    mine = uc_min_times(up, down)
    ref = pypsa_day(min_up=up, min_down=down)
    assert np.isclose(mine["cost"], ref.objective, rtol=1e-9), name
    rows[name] = {"cost": mine["cost"], "extra_cost": mine["cost"] - n_uc.objective,
                  "ccgt_hours_on": int(mine["status"]["ccgt"].sum())}
pd.DataFrame(rows).T.round(0)

**7. Interpretation**

The 6-hour limits change nothing; the 12-hour limits cost tens of thousands of euros. Why? And PyPSA needs `up_time_before` / `down_time_before`: what would go wrong if you told it the CCGT had been off for only one hour before the day, with a 6-hour minimum down time — and why does this matter when a model is run day after day?

*Your answer here.*